# 21_Packet_P004_Shortlist_Robustness
## Materia Arche Agent OS v3.0

**Work Packet ID:** P-004
**Decision this packet informs:** Is the NB17 lab panel robust — do the same top candidates appear across different train/test splits?
**Fastest falsifier:** Rank all test-set devices across 20 splits — if top candidates shuffle completely, the shortlist is fragile.
**Success threshold:** NB17 lab panel candidates appear in top-20 in ≥15/20 splits
**Failure / kill threshold:** Lab panel candidates appear in top-20 in <10/20 splits
**Reuse requirement if it fails:** Document which candidates are robust and which are not
**Dataset ID/version:** perovskite_stability_clean.csv (1,543 devices)
**Benchmark ID:** NB17 lab panel + P-003 uncertainty results
**Owner:** ML Scientist
**Reviewer:** Evidence Guardian
**Release ceiling:** Internal

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded — Packet P-004")

Libraries loaded — Packet P-004


## 1. Load data and locked model config

In [2]:
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Dataset: {len(df)} devices")

ml_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
               'MA', 'FA', 'Cs',
               'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
               'Perovskite_thickness', 'Cell_area_measured', 'JV_default_Voc',
               'JV_default_Jsc', 'JV_default_FF']

X = df[ml_features].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

# Locked ET config from NB15
et_params = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False)

print(f"Features: {len(ml_features)}")
print(f"Locked ET params: {et_params}")

Dataset: 1543 devices
Features: 16
Locked ET params: {'n_estimators': 700, 'max_features': 'sqrt', 'min_samples_split': 3, 'min_samples_leaf': 1, 'bootstrap': False}


## 2. Identify NB17 lab panel candidates

From NB17, the 3 candidates were selected on the frozen split (seed=42). Let's identify them and then check if they remain top-ranked across other splits.

In [3]:
# First, reproduce the frozen split ranking to identify our lab panel devices
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
et = ExtraTreesRegressor(random_state=42, **et_params)
et.fit(X_train, y_train)
pred = et.predict(X_test)

# Build ranking on frozen split
frozen_ranking = pd.DataFrame({
    'actual_T80': np.expm1(y_test.values),
    'predicted': np.expm1(pred),
}, index=X_test.index)
frozen_ranking['rank'] = frozen_ranking['predicted'].rank(ascending=False).astype(int)
frozen_ranking = frozen_ranking.sort_values('rank')

# Show top 10 from frozen split
print("Top 10 from frozen split (seed=42):")
print(frozen_ranking.head(10).to_string())
print("")

# Identify the top-ranked devices (these are the candidate pool)
top10_frozen_indices = frozen_ranking.head(10).index.tolist()
top3_frozen_indices = frozen_ranking.head(3).index.tolist()
print(f"Top 3 device indices (frozen): {top3_frozen_indices}")
print(f"Top 10 device indices (frozen): {top10_frozen_indices}")

Top 10 from frozen split (seed=42):
       actual_T80    predicted  rank
1063  5400.000000  2981.596091     1
1374  1400.000000  2235.202608     2
289   2300.000000  1039.710130     3
1417   500.000000   851.014700     4
898     40.000000   740.038455     5
590     13.590287   674.177686     6
1500  2400.000000   590.518830     7
610    806.000000   547.234902     8
485   2800.000000   534.760707     9
107    700.000000   502.127449    10

Top 3 device indices (frozen): [1063, 1374, 289]
Top 10 device indices (frozen): [1063, 1374, 289, 1417, 898, 590, 1500, 610, 485, 107]


## 3. Track top candidates across 20 random splits

For each split, train the locked ET model and rank ALL devices in the dataset (not just the test set). This lets us track how individual devices move in rank.

In [4]:
n_splits = 20
seeds = list(range(1, n_splits + 1))

# For each split, train on 80% and predict on ALL devices
# Track how often each device appears in top-20 when it's in the test set
device_top20_counts = Counter()  # device_idx -> times in top-20 of test set
device_appearances = Counter()   # device_idx -> times in test set

# Also track full-dataset rankings
all_ranks = {idx: [] for idx in range(len(df))}

for seed in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    et = ExtraTreesRegressor(random_state=42, **et_params)
    et.fit(X_tr, y_tr)
    
    # Predict on full dataset
    pred_all = et.predict(X)
    ranks_all = pd.Series(pred_all, index=X.index).rank(ascending=False).astype(int)
    
    for idx in range(len(df)):
        all_ranks[idx].append(ranks_all.iloc[idx])
    
    # Track test-set top-20
    pred_test = et.predict(X_te)
    test_ranking = pd.Series(pred_test, index=X_te.index).rank(ascending=False).astype(int)
    top20_test = test_ranking[test_ranking <= 20].index.tolist()
    
    for idx in X_te.index:
        device_appearances[idx] += 1
    for idx in top20_test:
        device_top20_counts[idx] += 1

print(f"Completed {n_splits} splits")
print(f"Devices tracked: {len(all_ranks)}")

Completed 20 splits
Devices tracked: 1543


## 4. Rank stability analysis

In [5]:
# Compute rank statistics for each device
rank_stats = pd.DataFrame({
    'actual_T80': np.expm1(y.values),
    'mean_rank': [np.mean(all_ranks[i]) for i in range(len(df))],
    'std_rank': [np.std(all_ranks[i]) for i in range(len(df))],
    'min_rank': [np.min(all_ranks[i]) for i in range(len(df))],
    'max_rank': [np.max(all_ranks[i]) for i in range(len(df))],
})

# Sort by mean rank
rank_stats = rank_stats.sort_values('mean_rank')

print("=" * 80)
print("TOP 20 MOST CONSISTENTLY HIGH-RANKED DEVICES (across 20 splits)")
print("=" * 80)
print(f"{'Idx':>5} {'Actual T80':>10} {'Mean Rank':>10} {'Std':>8} {'Best':>6} {'Worst':>6}")
print("-" * 80)
for i, (idx, row) in enumerate(rank_stats.head(20).iterrows()):
    frozen_marker = " <-- frozen top-3" if idx in top3_frozen_indices else ""
    print(f"{idx:>5} {row['actual_T80']:>10.0f} {row['mean_rank']:>10.1f} {row['std_rank']:>8.1f} {row['min_rank']:>6.0f} {row['max_rank']:>6.0f}{frozen_marker}")

TOP 20 MOST CONSISTENTLY HIGH-RANKED DEVICES (across 20 splits)
  Idx Actual T80  Mean Rank      Std   Best  Worst
--------------------------------------------------------------------------------
  850       3400       11.2     12.0      2     62
  848       2200       13.9      5.8      6     28
  849       3400       14.8     17.7      1     80
 1066       5400       15.3     50.0      1    230
 1064       5400       16.2     43.3      2    203
  399       2586       16.2     12.6      6     57
 1065       5400       18.5     66.9      1    309
  631       2500       20.6     17.6      9     65
  398       2240       20.8      9.5      9     43
 1379       1824       23.7      4.6     15     32
 1380       1824       23.7      4.6     15     32
  485       2800       24.1     39.7     10    197
 1063       5400       24.6     79.3      1    366 <-- frozen top-3
  630       2500       24.7     25.9      7     94
 1374       1400       27.8     23.6      8     68 <-- frozen top-3
  397

## 5. Check frozen top-3 stability

In [6]:
print("=" * 65)
print("FROZEN TOP-3 RANK STABILITY")
print("=" * 65)

n_in_top20 = 0
for idx in top3_frozen_indices:
    stats = rank_stats.loc[idx]
    in_top20 = stats['mean_rank'] <= 20
    if in_top20:
        n_in_top20 += 1
    marker = "IN TOP-20" if in_top20 else "NOT in top-20"
    print(f"  Device {idx}: actual T80={stats['actual_T80']:.0f}h")
    print(f"    Mean rank: {stats['mean_rank']:.1f} (std {stats['std_rank']:.1f})")
    print(f"    Range: [{stats['min_rank']:.0f}, {stats['max_rank']:.0f}]")
    print(f"    Status: {marker}")
    print("")

print(f"Summary: {n_in_top20}/3 frozen candidates consistently in top-20")
print("")

# Also check: which devices are MOST consistently in top-20 of test sets?
print("=" * 65)
print("MOST FREQUENTLY TOP-20 IN TEST SETS")
print("=" * 65)
freq_top20 = []
for idx, count in device_top20_counts.most_common(20):
    appearances = device_appearances[idx]
    rate = count / appearances * 100
    actual = np.expm1(y.iloc[idx])
    frozen_marker = " *" if idx in top3_frozen_indices else ""
    freq_top20.append({'idx': idx, 'top20_count': count, 'appearances': appearances,
                       'rate': rate, 'actual_T80': actual})
    print(f"  Device {idx:>4}: top-20 in {count}/{appearances} test appearances ({rate:.0f}%), actual T80={actual:.0f}h{frozen_marker}")

print("\n  * = frozen top-3 candidate")

FROZEN TOP-3 RANK STABILITY
  Device 1063: actual T80=5400h
    Mean rank: 24.6 (std 79.3)
    Range: [1, 366]
    Status: NOT in top-20

  Device 1374: actual T80=1400h
    Mean rank: 27.8 (std 23.6)
    Range: [8, 68]
    Status: NOT in top-20

  Device 289: actual T80=2300h
    Mean rank: 47.1 (std 78.8)
    Range: [11, 379]
    Status: NOT in top-20

Summary: 0/3 frozen candidates consistently in top-20

MOST FREQUENTLY TOP-20 IN TEST SETS
  Device  267: top-20 in 7/7 test appearances (100%), actual T80=600h
  Device  677: top-20 in 7/10 test appearances (70%), actual T80=61h
  Device 1380: top-20 in 7/7 test appearances (100%), actual T80=1824h
  Device  326: top-20 in 7/8 test appearances (88%), actual T80=500h
  Device  966: top-20 in 6/6 test appearances (100%), actual T80=100h
  Device 1066: top-20 in 6/6 test appearances (100%), actual T80=5400h
  Device 1373: top-20 in 6/6 test appearances (100%), actual T80=5000h
  Device  481: top-20 in 5/6 test appearances (83%), actual T

## 6. Final uncertainty-aware shortlist

In [7]:
# Produce final shortlist: top devices by mean rank with uncertainty context
shortlist = rank_stats.head(10).copy()
shortlist['rank_range'] = shortlist.apply(lambda r: f"[{r['min_rank']:.0f}, {r['max_rank']:.0f}]", axis=1)

# Add P-003 context: per-tree prediction interval from frozen split
et_frozen = ExtraTreesRegressor(random_state=42, **et_params)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
et_frozen.fit(X_tr, y_tr)
tree_preds_all = np.array([tree.predict(X) for tree in et_frozen.estimators_])
pred_p10_all = np.expm1(np.percentile(tree_preds_all, 10, axis=0))
pred_p90_all = np.expm1(np.percentile(tree_preds_all, 90, axis=0))
pred_mean_all = np.expm1(np.mean(tree_preds_all, axis=0))

shortlist['pred_mean_hours'] = [pred_mean_all[idx] for idx in shortlist.index]
shortlist['pred_p10_hours'] = [pred_p10_all[idx] for idx in shortlist.index]
shortlist['pred_p90_hours'] = [pred_p90_all[idx] for idx in shortlist.index]

print("=" * 85)
print("UNCERTAINTY-AWARE SHORTLIST (top 10 by cross-split rank stability)")
print("=" * 85)
for i, (idx, row) in enumerate(shortlist.iterrows(), 1):
    frozen = " [FROZEN TOP-3]" if idx in top3_frozen_indices else ""
    print(f"  #{i:>2} Device {idx:>4}{frozen}")
    print(f"      Actual T80: {row['actual_T80']:.0f}h")
    print(f"      Mean rank: {row['mean_rank']:.1f} +/- {row['std_rank']:.1f} {row['rank_range']}")
    print(f"      Prediction: {row['pred_mean_hours']:.0f}h (80% CI: {row['pred_p10_hours']:.0f}–{row['pred_p90_hours']:.0f}h)")
    print("")

shortlist_export = shortlist[['actual_T80', 'mean_rank', 'std_rank', 'min_rank', 'max_rank',
                              'pred_mean_hours', 'pred_p10_hours', 'pred_p90_hours']].round(1)
shortlist_export.to_csv("Packet_P004_Robust_Shortlist.csv")
print("Packet_P004_Robust_Shortlist.csv exported")

UNCERTAINTY-AWARE SHORTLIST (top 10 by cross-split rank stability)
  # 1 Device  850
      Actual T80: 3400h
      Mean rank: 11.2 +/- 12.0 [2, 62]
      Prediction: 3118h (80% CI: 2735–3400h)

  # 2 Device  848
      Actual T80: 2200h
      Mean rank: 13.9 +/- 5.8 [6, 28]
      Prediction: 2214h (80% CI: 1795–2735h)

  # 3 Device  849
      Actual T80: 3400h
      Mean rank: 14.8 +/- 17.7 [1, 80]
      Prediction: 3231h (80% CI: 3400–3400h)

  # 4 Device 1066
      Actual T80: 5400h
      Mean rank: 15.3 +/- 50.0 [1, 230]
      Prediction: 5392h (80% CI: 5400–5400h)

  # 5 Device 1064
      Actual T80: 5400h
      Mean rank: 16.2 +/- 43.3 [2, 203]
      Prediction: 5027h (80% CI: 5400–6735h)

  # 6 Device  399
      Actual T80: 2586h
      Mean rank: 16.2 +/- 12.6 [6, 57]
      Prediction: 2264h (80% CI: 2034–2586h)

  # 7 Device 1065
      Actual T80: 5400h
      Mean rank: 18.5 +/- 66.9 [1, 309]
      Prediction: 5295h (80% CI: 5400–5400h)

  # 8 Device  631
      Actual T80: 2500h


## 7. Honest status footer

In [8]:
# Determine status
if n_in_top20 == 3:
    status = "Confirmed"
    decision = "Advance"
elif n_in_top20 >= 2:
    status = "Promising"
    decision = "Iterate"
elif n_in_top20 >= 1:
    status = "Promising"
    decision = "Iterate"
else:
    status = "Negative"
    decision = "Stop"

print("=" * 65)
print("HONEST STATUS — MATERIA ARCHE v3.0")
print("=" * 65)
print(f"Packet ID: P-004")
print(f"Status: {status}")
print(f"Evidence level: E3 (decision-grade shortlist)")
print(f"Decision outcome: {decision}")
print(f"Release ceiling: Internal")
print(f"Domain: perovskite")
print(f"Dataset version: perovskite_stability_clean.csv (1,543 devices)")
print(f"Benchmark: NB17 frozen top-3 lab panel")
print(f"This run: {n_in_top20}/3 frozen candidates stable in top-20 across 20 splits")

if n_in_top20 == 3:
    print(f"What worked: All 3 lab panel candidates are rank-stable")
    print(f"What failed or remains uncertain: Prediction intervals remain wide (P-003)")
    print(f"What changes next: Lab panel is confirmed — proceed to Phase 2 with uncertainty context")
elif n_in_top20 >= 1:
    print(f"What worked: {n_in_top20}/3 candidates are rank-stable")
    print(f"What failed or remains uncertain: {3 - n_in_top20}/3 candidates not consistently top-ranked")
    print(f"What changes next: Consider replacing unstable candidates with more robust alternatives")
else:
    print(f"What worked: Identified which devices are truly rank-stable")
    print(f"What failed or remains uncertain: Frozen top-3 are not robust across splits")
    print(f"What changes next: Select new candidates from cross-split stable rankings")

print(f"Reusable asset created: Packet_P004_Robust_Shortlist.csv")
print(f"Safeguard added: Cross-split rank stability check for all future shortlists")
print(f"")
print(f"Reviewer sign-off: Evidence Guardian __________")

HONEST STATUS — MATERIA ARCHE v3.0
Packet ID: P-004
Status: Negative
Evidence level: E3 (decision-grade shortlist)
Decision outcome: Stop
Release ceiling: Internal
Domain: perovskite
Dataset version: perovskite_stability_clean.csv (1,543 devices)
Benchmark: NB17 frozen top-3 lab panel
This run: 0/3 frozen candidates stable in top-20 across 20 splits
What worked: Identified which devices are truly rank-stable
What failed or remains uncertain: Frozen top-3 are not robust across splits
What changes next: Select new candidates from cross-split stable rankings
Reusable asset created: Packet_P004_Robust_Shortlist.csv
Safeguard added: Cross-split rank stability check for all future shortlists

Reviewer sign-off: Evidence Guardian __________
